# Delivery Performance Analysis

## Goal

This notebook analyzes delivery performance based on the available logistics data. The 
goal is to examine delivery times and delays, and to identify potential differences 
between regions and transport types.

The following questions are examined:

1. What is the average delivery duration?
2. How often are deliveries delayed?
3. Which regions have particularly long delivery times?
4. Which equipment types are particularly fast or slow?
5. Are there specific regions with an above-average number of delays?

To answer these questions, the relevant tables are first merged, then cleaned and 
analyzed. The results are presented through appropriate visualizations.

## Data Basis

The analysis primarily uses the following tables:

- `loads` - information about the shipments
- `trips` - information about the trips carried out
- `delivery_events` - information about pickup and delivery
- `routes` - information about the respective routes

The tables are related as follows:

- `delivery_events`-(`trip_id`)->`trips`-(`load_id`)->`loads`-(`route_id`)->`routes`
- `delivery_events`-(`load_id`)->`loads`

Data cleaning, merging, and metric calculation (`delivery_duration_hours`, 
`delivery_delay_hours`, `is_delayed_delivery`) happen in `src/delivery_pipeline.py`, 
not in this notebook. This notebook loads the already-cleaned result and focuses on 
analysis and visualization.

*Note on charts: **Matplotlib** is used as the base for most charts. For the 
distribution and scatter plots, **Seaborn** is used on top of it, since it provides 
built-in support for density curves and bubble-size legends without additional manual 
calculation.*

## 1. Load Data (via Pipeline)

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from delivery_pipeline import run_pipeline

clean_df = run_pipeline()
print(f"Loads used for analysis: {len(clean_df)}")


*Note on data cleaning: Loads where the actual delivery timestamp is before the actual 
pickup timestamp were already excluded in the pipeline (see 
`drop_inconsistent_timestamps()` in `delivery_pipeline.py`). The raw data itself is not 
modified; less than 1% of loads were affected.*

### Data Quality Note

During timestamp validation, an edge case was found: one delivery had a rounded 
`delivery_duration_hours` value of 0.0, even though the exact timestamp comparison 
showed a (minimal) negative difference between the delivery and pickup timestamps. A 
plain filter on `delivery_duration_hours >= 0` would have let this inconsistent record 
pass as valid.

The check is therefore consistently performed on the unrounded timestamps 
(`delivery_actual_datetime >= pickup_actual_datetime`), not on the already rounded, 
derived metric, see `drop_inconsistent_timestamps()` in `delivery_pipeline.py`.

## 2. Delivery Duration and Punctuality

### 2.1 What is the average delivery duration?

In [ ]:
delayed_trips = clean_df[clean_df["is_delayed_delivery"]]
share_delayed = clean_df["is_delayed_delivery"].mean() * 100

print("Delivery duration:")
display(clean_df["delivery_duration_hours"].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(clean_df["delivery_duration_hours"], bins=30, kde=True, ax=ax)
ax.axvline(
    clean_df["delivery_duration_hours"].median(),
    color="red", linestyle="--",
    label=f"Median: {clean_df['delivery_duration_hours'].median():.1f}h",
)
ax.set_xlabel("Delivery duration (hours)")
ax.set_ylabel("Number of deliveries")
ax.set_title("Distribution of delivery duration")
ax.legend()
plt.show()

*The distribution of delivery duration is moderately right-skewed (mean: 26.68h vs. 
median: 24.88h), a small number of long-running deliveries pull the average upward. 
The middle 50% of deliveries (IQR) fall between 14.62h and 38.53h, while the overall 
range spans from near-instant to 73.77h, indicating some notable outliers worth 
investigating further.*

### 2.2 How often are deliveries delayed?

In [ ]:
print(f"Punctuality ({len(clean_df)} deliveries total):")
delayed_count = clean_df["is_delayed_delivery"].sum()
print(f"Delayed deliveries: {delayed_count} ({share_delayed:.2f}%)")

In [ ]:
delay_counts = clean_df["is_delayed_delivery"].value_counts()
labels = ["On-time / early", "Delayed"]
values = [delay_counts.get(False, 0), delay_counts.get(True, 0)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values)
ax.bar_label(bars, labels=[f"{v} ({v/sum(values)*100:.1f}%)" for v in values])
ax.set_ylabel("Number of deliveries")
ax.set_title("Delivery punctuality")
plt.show()

*67.04% of deliveries (56,933 of 84,924) were delayed, the majority of the dataset, not 
an edge case. Note that this uses a strict definition (`delivery_delay_hours > 0`, i.e. 
any delay counts, with no buffer or tolerance window). This high rate makes on-time 
performance a central issue across the dataset rather than an isolated problem.*

### 2.3 How large are the delays?

In [ ]:
print("Delay duration (delayed deliveries only):")
display(delayed_trips["delivery_delay_hours"].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(delayed_trips["delivery_delay_hours"], bins=30)
ax.set_xlabel("Delay (hours)")
ax.set_ylabel("Number of deliveries")
ax.set_title("Distribution of delay duration (delayed deliveries only)")
plt.show()

*Unlike delivery duration, the distribution of delay length is nearly symmetric (mean 
and median both 3.00h), with most delayed deliveries falling between 1.51h and 4.51h 
(IQR). The maximum delay is exactly 6.00h across the entire dataset, suggesting a fixed 
upper bound in how the (synthetic) data was generated rather than an operational cap.*

## 3. Regional Differences

### 3.1 Which regions have particularly long delivery times and an above-average share of delays?

In [ ]:
region_stats = (
    clean_df.groupby("delivery_location_state")
    .agg(
        avg_duration_hours=("delivery_duration_hours", "mean"),
        delayed_share=("is_delayed_delivery", "mean"),
        avg_delay_hours=("delivery_delay_hours", lambda s: s[s > 0].mean()),
        n_loads=("load_id", "size"),
    )
    .round(2)
    .sort_values("delayed_share", ascending=False)
)
display(region_stats)

*Average delivery duration varies substantially by region, from 11.86h (TN) to 
40.40h (WA), more than a 3x difference. Delay frequency (66%-71%) and delay severity 
(2.95h-3.07h) are far more consistent across regions by comparison. This suggests 
regional duration differences are primarily driven by distance/lane characteristics, 
while delays stem from a more uniform, systemic factor (see 3.2 below). Note that 
smaller regions (e.g. IL, NV with under 1,500 loads) carry more statistical uncertainty 
than larger ones (e.g. CA with ~8,900 loads).*

### 3.2 Delay frequency vs. delay severity

The two metrics `delayed_share` and `avg_delay_hours` can be visualized separately 
(clear per-metric comparison) or combined (to spot regions that struggle on both 
dimensions at once). Both views are shown below.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, max(4, len(region_stats) * 0.4)))
ax[0].barh(region_stats.index, region_stats["delayed_share"] * 100)
ax[0].set_xlabel("Share of delayed deliveries (%)")
ax[0].set_title("Delay rate by region")
ax[0].invert_yaxis() 

ax[1].barh(region_stats.index, region_stats["avg_delay_hours"], color="tab:orange")
ax[1].set_xlabel("Avg. delay (hours)")
ax[1].set_title("Average delay severity by region")
ax[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
plot_df = region_stats.reset_index()
plot_df["delayed_share_pct"] = plot_df["delayed_share"] * 100

fig, ax = plt.subplots(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x="delayed_share_pct",
    y="avg_delay_hours",
    size="n_loads", sizes=(50, 500),
    alpha=0.6, ax=ax, legend="brief",
)

for state, row in region_stats.iterrows():
    ax.annotate(state, (row["delayed_share"] * 100, row["avg_delay_hours"]), fontsize=8)

legend = ax.get_legend()
legend.set_bbox_to_anchor((1.02, 1))  
for text in legend.get_texts():
    text.set_fontsize(9)
ax.legend(labelspacing=1.8, borderpad=1.2)

ax.set_xlabel("Share of delayed deliveries (%)")
ax.set_ylabel("Average delay (hours, delayed deliveries only)")
ax.set_title("Delay frequency vs. delay severity by region")
plt.tight_layout()
plt.show()

*The scatter plot shows that delay severity (avg_delay_hours) is remarkably consistent 
across regions, ranging only from 2.95h to 3.07h, a difference of about 7 minutes. 
Delay frequency (delayed_share), on the other hand, varies more meaningfully (66%-71%), 
with NY showing the highest rate. This suggests that once a delivery is delayed, the 
typical delay length is largely independent of region, but the likelihood of being 
delayed at all is somewhat higher in NY than elsewhere, a pattern also visible in the 
delay-rate bar chart above.*

## 4. Differences by Equipment Type

### 4.1 Which equipment types (Dry Van vs. Refrigerated) show different delivery durations or delay rates?

In [ ]:
transport_column = "load_type" 

mode_stats = (
    clean_df.groupby(transport_column)
    .agg(
        avg_duration_hours=("delivery_duration_hours", "mean"),
        delayed_share=("is_delayed_delivery", "mean"),
        n_loads=("load_id", "size"),
    )
    .round(2)
    .sort_values("delayed_share", ascending=False)
)  
display(mode_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, max(4, len(mode_stats) * 0.4)))

axes[0].barh(mode_stats.index, mode_stats["avg_duration_hours"])
axes[0].set_xlabel("Avg. delivery duration (hours)")
axes[0].set_title("Delivery duration by transport type")
axes[0].invert_yaxis()

axes[1].barh(mode_stats.index, mode_stats["delayed_share"] * 100, color="tab:orange")
axes[1].set_xlabel("Share of delayed deliveries (%)")
axes[1].set_title("Delay rate by transport type")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

*Delivery duration and delay rate show no meaningful difference between equipment types 
(Dry Van: 26.70h / 67% delayed vs. Refrigerated: 26.66h / 67% delayed). Equipment type 
does not appear to be a relevant driver of delivery performance in this dataset, other 
factors (e.g. region, as shown above) have a much larger effect.*

## 5. Conclusion

With 67% of deliveries delayed under a strict "any delay counts" definition 
(delivery_delay_hours > 0, no buffer), on-time performance is a clear area for 
improvement across the dataset, not isolated to specific regions or equipment types 
alone.

Regional differences turned out to be more nuanced than expected: while delivery 
*duration* varies strongly by region (11.86h-40.40h), delay *frequency* and *severity* 
are remarkably uniform (66%-71% and 2.95h-3.07h respectively). This points to two 
distinct underlying mechanisms, distance/lane characteristics driving duration, and a 
more systemic, region-independent factor driving delays. Equipment type, by contrast, 
showed no measurable effect on either metric.